# 03 — Week 1 acceptance and truth lock


Prove that translator output is identical after evaluation inputs—including
`tickets.csv`—are removed. A timestamp canary and a deliberately leaky negative
control prove the harness can detect value-level leakage. Runtime mount tests
are secondary deployment evidence, not the primary proof.


    This is a **thin orchestration notebook**. The tested implementation lives
    in the GitHub package; this notebook only sets paths, calls one workflow,
    and displays its evidence. Run cells from top to bottom.

In [ ]:
# Shared implementation: GitHub in Colab, local source when testing this repository.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/LiliDopidze/anomaly_detection.git"
RUNTIME_REF = os.getenv("ANOMALY_RUNTIME_REF", "main")
LOCAL_SOURCE = os.getenv("ANOMALY_SOURCE_ROOT")

if LOCAL_SOURCE:
    sys.path.insert(0, str(Path(LOCAL_SOURCE).resolve()))
    RUNTIME_COMMIT = "local-working-tree"
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        f"git+{REPOSITORY}@{RUNTIME_REF}",
    ])
    RUNTIME_COMMIT = subprocess.check_output(
        ["git", "ls-remote", REPOSITORY, RUNTIME_REF], text=True
    ).split()[0]

os.environ["ANOMALY_RUNTIME_REF"] = RUNTIME_REF
os.environ["ANOMALY_RUNTIME_COMMIT"] = RUNTIME_COMMIT
print(f"Runtime: {RUNTIME_REF} ({RUNTIME_COMMIT[:12]})")

In [ ]:
# Mount Google Drive in Colab. Local validation skips this block.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
print(f"Drive root: {DRIVE_ROOT}")

## Configuration

Use the same telecom source and `RUN_ID` that completed in Notebook 02.
This notebook reads that immutable run and adds `acceptance_report.json`.

In [ ]:
TELECOM_SOURCE = Path(os.getenv(
    "ANOMALY_TELECOM_SOURCE",
    str(DRIVE_ROOT),
))
OUTPUT_ROOT = Path(os.getenv(
    "ANOMALY_OUTPUT_ROOT",
    str(DRIVE_ROOT / "outputs" / "milestone_1" / "v0.3" / "telecom"),
))
RUN_ID = os.getenv("ANOMALY_RUN_ID", "telecom_full_v1")
RUN_ROOT = OUTPUT_ROOT / RUN_ID

print(f"Source: {TELECOM_SOURCE}")
print(f"Existing Notebook 02 run: {RUN_ROOT}")

In [ ]:
from anomaly_detection.workflows import run_week1_acceptance

acceptance = run_week1_acceptance(TELECOM_SOURCE, RUN_ROOT)

In [ ]:
from pprint import pprint

pprint({
    "translator_invariance": acceptance["translator_invariance"]["passed"],
    "value_leakage_test": acceptance["value_level_leakage"]["passed"],
    "negative_control_detected":
        acceptance["value_level_leakage"]["negative_control_detected"],
    "runtime_isolation": acceptance["runtime_isolation"]["passed"],
    "delayed_known_at_guard": acceptance["as_of_guard"]["passed"],
    "quality_counts":
        acceptance["materialised_telemetry"]["quality_counts"],
    "exposure_ranges":
        acceptance["materialised_telemetry"]["exposure_ranges"],
})
print(f"Evidence: {RUN_ROOT / 'acceptance_report.json'}")